# CatBoost + Optuna + SHAP（可読性改善版）

このノートブックは、Excelから読み込んだ特徴量データを前処理し、CatBoost分類器をOptunaでチューニングした上で、SHAPで特徴量重要度を可視化します。


In [ ]:
# --- Imports ---
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path

import json
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import shap
import optuna
from optuna.integration import OptunaSearchCV

from catboost import CatBoostClassifier
from joblib import dump
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix


In [ ]:
# --- Config ---
@dataclass(frozen=True)
class Config:
    data_path: Path = Path("./1.2 脳血流と問題カテゴリの関連 - 背外側追加.xlsx")
    sheet_name: str = "ディアビアイ"

    target_col: str = "score"
    drop_cols: tuple[str, ...] = ("subject", "question", "q_category", "dataset_no", "sheet_name")

    # 例：オキシヘモグロビンのみ使う / 主要特徴量のみ残す、など
    drop_feature_cols: tuple[str, ...] = (
        # 標準偏差系（.1/.2）を落とす例
        "CH1.1_std","CH2.1_std","CH3.1_std","CH4.1_std","CH5.1_std","CH6.1_std","CH7.1_std","CH8.1_std",
        "CH9.1_std","CH10.1_std","CH11.1_std","CH12.1_std","CH13.1_std","CH14.1_std","CH15.1_std","CH16.1_std",
        "CH17.1_std","CH18.1_std","CH19.1_std","CH20.1_std","CH21.1_std","CH22.1_std",
        "CH1.2_std","CH2.2_std","CH3.2_std","CH4.2_std","CH5.2_std","CH6.2_std","CH7.2_std","CH8.2_std",
        "CH9.2_std","CH10.2_std","CH11.2_std","CH12.2_std","CH13.2_std","CH14.2_std","CH15.2_std","CH16.2_std",
        "CH17.2_std","CH18.2_std","CH19.2_std","CH20.2_std","CH21.2_std","CH22.2_std",

        # 主要特徴量だけ残す場合に落とす例（必要に応じて調整）
        "right_pupil_std",
        "CH1_std","CH2_std","CH3_std","CH4_std","CH5_std","CH6_std","CH7_std","CH8_std","CH9_std","CH10_std","CH11_std",
        "CH12_std","CH13_std","CH14_std","CH15_std","CH16_std","CH17_std","CH18_std","CH19_std","CH20_std","CH21_std","CH22_std",
    )

    test_size: float = 0.2
    random_state: int = 42

    # OptunaSearchCV
    n_splits: int = 2
    n_trials: int = 50

    # 出力保存（必要ならパスを変更）
    out_dir: Path = Path("./results")

CFG = Config()
CFG


In [ ]:
# --- Helper functions ---
def sanitize_columns(columns: pd.Index) -> list[str]:
    """SHAP / 可視化で扱いやすいように列名を安全な文字に変換します。"""
    cleaned = []
    for col in columns.astype(str):
        col = (
            col.replace(":", "_")
               .replace("/", "_")
               .replace("[", "")
               .replace("]", "")
               .replace(" ", "_")
               .replace(".", "_")
               .replace(",", "_")
               .replace("(", "_")
               .replace(")", "_")
        )
        cleaned.append(col)
    return cleaned


def drop_existing(df: pd.DataFrame, cols: tuple[str, ...]) -> pd.DataFrame:
    """存在する列だけを安全に drop します（列が無くても落ちない）。"""
    existing = [c for c in cols if c in df.columns]
    return df.drop(columns=existing)


def ensure_out_dir(out_dir: Path) -> Path:
    out_dir.mkdir(parents=True, exist_ok=True)
    return out_dir


In [ ]:
# --- Load data ---
assert CFG.data_path.exists(), f"File not found: {CFG.data_path.resolve()}"

df = pd.read_excel(CFG.data_path, sheet_name=CFG.sheet_name)
print(df.shape)
df.head()


In [ ]:
# --- Preprocess ---
# 1) 目的変数の変換（例：2→1）
if CFG.target_col in df.columns:
    df[CFG.target_col] = df[CFG.target_col].replace(2, 1)

# 2) 不要列を削除
df = drop_existing(df, CFG.drop_cols)

# 3) 特徴量列を削除（必要に応じて CFG.drop_feature_cols を編集）
df = drop_existing(df, CFG.drop_feature_cols)

# 4) 列名をサニタイズ
df.columns = sanitize_columns(df.columns)

print(df.shape)
df.head()


In [ ]:
# --- (Optional) Correlation heatmap ---
# 変数が多い場合は非常に重くなるので、必要なときだけ実行してください。
show_heatmap = False

if show_heatmap:
    corr = df.corr(numeric_only=True)
    plt.figure(figsize=(30, 24))
    sns.heatmap(corr, annot=False, fmt=".2f", square=True, linewidths=0.2)
    plt.title("Correlation Matrix")
    plt.tight_layout()
    plt.show()


In [ ]:
# --- Train / Test split ---
X = df.drop(columns=[CFG.target_col])
y = df[CFG.target_col]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=CFG.test_size,
    random_state=CFG.random_state,
    stratify=y
)

print(X_train.shape, X_test.shape)


In [ ]:
# --- Baseline model (optional) ---
baseline_model = CatBoostClassifier(random_state=CFG.random_state, verbose=0)
baseline_model.fit(X_train, y_train)

baseline_pred = baseline_model.predict(X_test)
print("Baseline accuracy:", accuracy_score(y_test, baseline_pred))


In [ ]:
# --- Hyperparameter tuning with OptunaSearchCV ---
param_dist = {
    "iterations": optuna.distributions.IntDistribution(200, 1000),
    "depth": optuna.distributions.IntDistribution(3, 10),
    "learning_rate": optuna.distributions.FloatDistribution(1e-3, 0.3, log=True),
    "l2_leaf_reg": optuna.distributions.FloatDistribution(1.0, 10.0),
    "bagging_temperature": optuna.distributions.FloatDistribution(0.0, 1.0),
    "border_count": optuna.distributions.IntDistribution(32, 255),
}

cv = StratifiedKFold(n_splits=CFG.n_splits, shuffle=True, random_state=CFG.random_state)

optuna_search = OptunaSearchCV(
    estimator=CatBoostClassifier(random_state=CFG.random_state, verbose=0),
    param_distributions=param_dist,
    cv=cv,
    n_trials=CFG.n_trials,
    scoring="accuracy",
    n_jobs=-1,
    verbose=1,
)

optuna_search.fit(X_train, y_train)

best_model = optuna_search.best_estimator_
optuna_search.best_params_


In [ ]:
# --- Evaluate ---
y_pred = best_model.predict(X_test)

acc_test = accuracy_score(y_test, y_pred)
acc_train = accuracy_score(y_train, best_model.predict(X_train))

print("Best params:", optuna_search.best_params_)
print(f"Accuracy (test) : {acc_test:.4f}")
print(f"Accuracy (train): {acc_train:.4f}")

print("\nConfusion matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification report:")
print(classification_report(y_test, y_pred))


In [ ]:
# --- SHAP ---
# TreeExplainer は CatBoost に対応
explainer = shap.TreeExplainer(best_model)

# shap_values: (n_samples, n_features) もしくはクラスごと
shap_values = explainer.shap_values(X)

# newer API (Explanation)
explanation = explainer(X)

type(shap_values), explanation.shape


In [ ]:
# --- SHAP plots ---
# 目的に応じて必要なものだけ実行してください。

# (1) beeswarm
shap.plots.beeswarm(explanation, max_display=20)

# (2) bar（平均絶対値）
shap.plots.bar(explanation.abs.mean(0), max_display=20)

# (3) heatmap
shap.plots.heatmap(explanation, max_display=12)

# (4) waterfall（1サンプル）
shap.plots.waterfall(explanation[0])


In [ ]:
# --- Save artifacts (optional) ---
do_save = False

if do_save:
    out_dir = ensure_out_dir(CFG.out_dir)
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

    # 1) model / params / metrics
    dump(best_model, out_dir / f"model_{timestamp}.joblib")
    with open(out_dir / f"params_{timestamp}.json", "w", encoding="utf-8") as f:
        json.dump(optuna_search.best_params_, f, ensure_ascii=False, indent=2)
    with open(out_dir / f"metrics_{timestamp}.json", "w", encoding="utf-8") as f:
        json.dump({"accuracy_test": acc_test, "accuracy_train": acc_train}, f, ensure_ascii=False, indent=2)

    # 2) SHAP values (二値分類ならクラス1を保存したいケースが多い)
    # shap_values の形式はモデル/SHAPのバージョンにより異なるため、分岐して保存
    if isinstance(shap_values, list) and len(shap_values) >= 2:
        pd.DataFrame(shap_values[1], columns=X.columns).to_csv(out_dir / f"shap_values_class1_{timestamp}.csv", index=False)
    else:
        pd.DataFrame(shap_values, columns=X.columns).to_csv(out_dir / f"shap_values_{timestamp}.csv", index=False)

    # 3) explainer
    dump(explainer, out_dir / f"shap_explainer_{timestamp}.joblib")

    print("Saved to:", out_dir.resolve())
